# SoundScape AI — Launch the Integrated Studio in Colab

This launches the **web platform** from `SoundScape_AI_Integrated_Studio.zip` and can optionally enable **fresh AudioLDM 2 generation** on a Colab T4 GPU. The five genuine WAV recordings and the six original/LoRA comparison clips are included in the ZIP; they play even if no checkpoint is uploaded.

**Before starting:** Runtime → Change runtime type → **T4 GPU**. Only upload checkpoint files created in your own trusted Colab sessions. Internet is needed for initial package/model downloads. The first generation request can be slow because the pretrained model is downloaded and cached.

## 1. Upload and extract the Studio ZIP

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # select SoundScape_AI_Integrated_Studio.zip
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives)==1, 'Upload exactly one SoundScape_AI_Integrated_Studio.zip'
with zipfile.ZipFile(archives[0]) as z:
    bad=[name for name in z.namelist() if name.startswith('/') or '..' in Path(name).parts]
    assert not bad, 'Unexpected archive member path'
    z.extractall('/content')
PROJECT=Path('/content/SoundScape_AI_Integrated_Studio')
assert (PROJECT/'app'/'main.py').exists(), 'Studio root folder was not found in the ZIP'
print('Project ready:',PROJECT)

## 2. Install dependencies (run once, restart if Colab requests it)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements-gpu.txt')],check=True)
print('Packages installed. If Colab requests a restart, restart once and continue with the next cell.')

## 3. Optional: upload model checkpoints

- `lora_adapters.pt` from Athir's Section 17 (`soundscape_export/extra/lora_adapters.pt`) enables the actual **Intensity** and **Comfort** adapters for *new* generation.
- `scene_mobilenetv3.pt` from your CV notebook enables the trained **scene classifier**.

Skip this cell if those files are not available yet: original AudioLDM 2 generation and playback of existing audio still work on GPU. **The `.ipynb` itself does not contain trained weights.**

In [ ]:
from google.colab import files
# Run this cell only if you have one or both .pt checkpoint files to upload.
weights=files.upload()
for name,data in weights.items():
    if name=='lora_adapters.pt':
        path=PROJECT/'models/audio/lora_adapters.pt'
    elif name=='scene_mobilenetv3.pt':
        path=PROJECT/'models/scene_mobilenetv3.pt'
    else:
        print('Not installed: unrecognized checkpoint',name)
        continue
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(data)
    print('Installed',path.name, 'bytes:',len(data))

## 4. Launch web studio

In [ ]:
import os,sys,threading,torch
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
os.environ['SOUNDSCAPE_ENABLE_GPU_AUDIO']='1' if torch.cuda.is_available() else '0'
print('CUDA available:',torch.cuda.is_available())
import uvicorn
from google.colab import output
server_thread=threading.Thread(target=lambda:uvicorn.run('app.main:app',host='0.0.0.0',port=8000,log_level='warning'),daemon=True)
server_thread.start()
print('Opening SoundScape Studio in a Colab browser window. Keep runtime connected.')
output.serve_kernel_port_as_window(8000)